# Section 1: Setup & Configuration

Load libraries, set evaluation date, and define paths.

In [1]:
import pandas as pd
import json
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Define evaluation date (fixed for demo consistency)

In [2]:
EVALUATION_DATE = datetime(2026, 4, 9)

In [3]:
print("=" * 60)
print("KYC COMPLETENESS CONTROL - DEMO ENVIRONMENT")
print("=" * 60)
print(f"\nEvaluation Date: {EVALUATION_DATE.strftime('%Y-%m-%d')}")
print("Status: Setup Complete ✓")

KYC COMPLETENESS CONTROL - DEMO ENVIRONMENT

Evaluation Date: 2026-04-09
Status: Setup Complete ✓


# Load Data Files

In [4]:
PROJECT_ROOT = Path().resolve()

RAW_DATA_PATH = PROJECT_ROOT / "Data Raw"
CLEAN_DATA_PATH = PROJECT_ROOT / "Data Clean"
CLEAN_DATA_PATH.mkdir(exist_ok=True)

# Load CSVs

In [5]:
customers_df = pd.read_csv(RAW_DATA_PATH / "customers.csv")
id_verifications_df = pd.read_csv(RAW_DATA_PATH / "id_verifications.csv")
documents_df = pd.read_csv(RAW_DATA_PATH / "documents.csv")
screenings_df = pd.read_csv(RAW_DATA_PATH / "screenings.csv")
transactions_df = pd.read_csv(RAW_DATA_PATH / "transactions.csv")

# Load JSONs

In [6]:
with open(RAW_DATA_PATH / "ubo.json", encoding="utf-8-sig") as f:
    ubo_data = json.load(f)
with open(RAW_DATA_PATH / "reg_changes.json", encoding="utf-8-sig") as f:
    reg_changes = json.load(f)

# Inform if data was loaded correctly

In [7]:
print("✓ customers.csv loaded:", customers_df.shape)
print("✓ id_verifications.csv loaded:", id_verifications_df.shape)
print("✓ documents.csv loaded:", documents_df.shape)
print("✓ screenings.csv loaded:", screenings_df.shape)
print("✓ transactions.csv loaded:", transactions_df.shape)
print("✓ ubo.json loaded:", len(ubo_data), "records")
print("✓ reg_changes.json loaded:", len(reg_changes), "rules")

✓ customers.csv loaded: (5120, 8)
✓ id_verifications.csv loaded: (3420, 6)
✓ documents.csv loaded: (7906, 4)
✓ screenings.csv loaded: (5037, 4)
✓ transactions.csv loaded: (5114, 3)
✓ ubo.json loaded: 1774 records
✓ reg_changes.json loaded: 4 rules


# Data Validation

In [8]:
print("=" * 60)
print("DATA VALIDATION CHECKS")
print("=" * 60)

DATA VALIDATION CHECKS


# Check for duplicates

In [9]:
print("\n1. DUPLICATE CHECKS:")
print(f"   Customers (unique): {customers_df['customer_id'].nunique()} / {len(customers_df)}")
print(f"   ID Verifications (unique): {id_verifications_df['customer_id'].nunique()} / {len(id_verifications_df)}")
print(f"   Documents (unique customers): {documents_df['customer_id'].nunique()} / {len(documents_df)}")
print(f"   Screenings (unique): {screenings_df['customer_id'].nunique()} / {len(screenings_df)}")


1. DUPLICATE CHECKS:
   Customers (unique): 5120 / 5120
   ID Verifications (unique): 3378 / 3420
   Documents (unique customers): 4579 / 7906
   Screenings (unique): 5037 / 5037


# Check for missing critical fields

In [10]:
print("\n2. MISSING DATA CHECKS:")
print(f"   Customers with NULL risk_rating: {customers_df['risk_rating'].isnull().sum()}")
print(f"   Customers with NULL jurisdiction: {customers_df['jurisdiction'].isnull().sum()}")
print(f"   ID Verifications with NULL expiry_date: {id_verifications_df['expiry_date'].isnull().sum()}")
print(f"   Screenings with NULL screening_date: {screenings_df['screening_date'].isnull().sum()}")


2. MISSING DATA CHECKS:
   Customers with NULL risk_rating: 10
   Customers with NULL jurisdiction: 7
   ID Verifications with NULL expiry_date: 0
   Screenings with NULL screening_date: 0


# Check date formats (convert to datetime)

In [11]:
print("\n3. DATE FORMAT CONVERSION:")
try:
    id_verifications_df['expiry_date'] = pd.to_datetime(id_verifications_df['expiry_date'])
    id_verifications_df['verification_date'] = pd.to_datetime(id_verifications_df['verification_date'])
    
    documents_df['expiry_date'] = pd.to_datetime(documents_df['expiry_date'])
    documents_df['upload_date'] = pd.to_datetime(documents_df['upload_date'])
    
    screenings_df['screening_date'] = pd.to_datetime(screenings_df['screening_date'])
    
    customers_df['last_kyc_review_date'] = pd.to_datetime(customers_df['last_kyc_review_date'], errors='coerce')
    customers_df['account_open_date'] = pd.to_datetime(customers_df['account_open_date'])
    
    transactions_df['last_txn_date'] = pd.to_datetime(transactions_df['last_txn_date'])
    
    print("   ✓ All date fields converted successfully")
except Exception as e:
    print(f"   ✗ Date conversion error: {e}")


3. DATE FORMAT CONVERSION:
   ✓ All date fields converted successfully


# Display sample rows

In [12]:
print("\n4. SAMPLE DATA:")
print("\nCustomers sample:")
print(customers_df[['customer_id', 'customer_name', 'jurisdiction', 'risk_rating']].head(3))
print("\nID Verifications sample:")
print(id_verifications_df[['customer_id', 'id_type', 'expiry_date']].head(3))
print("\nScreenings sample:")
print(screenings_df[['customer_id', 'screening_date', 'screening_result']].head(3))

print("\n✓ Data validation complete")


4. SAMPLE DATA:

Customers sample:
  customer_id               customer_name jurisdiction risk_rating
0        C001       Vertex Logistics GmbH           EU        High
1        C002   Sable Capital Partners LP           UK         Low
2        C003  Oakridge Technologies GmbH           US      Medium

ID Verifications sample:
  customer_id         id_type expiry_date
0        C001  Driver License  2026-04-24
1        C002     National ID  2025-09-24
2        C003  Driver License  2026-02-03

Screenings sample:
  customer_id screening_date screening_result
0        C001     2026-03-06           No Hit
1        C002     2026-02-19           No Hit
2        C003     2025-12-17           No Hit

✓ Data validation complete


# Write cleaned CSVs

In [13]:
customers_df.to_csv(CLEAN_DATA_PATH / "customers_clean.csv", index=False)
id_verifications_df.to_csv(CLEAN_DATA_PATH / "id_verifications_clean.csv", index=False)
documents_df.to_csv(CLEAN_DATA_PATH / "documents_clean.csv", index=False)
screenings_df.to_csv(CLEAN_DATA_PATH / "screenings_clean.csv", index=False)
transactions_df.to_csv(CLEAN_DATA_PATH / "transactions_clean.csv", index=False)

print(f"✓ customers_clean.csv written: {len(customers_df)} rows")
print(f"✓ id_verifications_clean.csv written: {len(id_verifications_df)} rows")
print(f"✓ documents_clean.csv written: {len(documents_df)} rows")
print(f"✓ screenings_clean.csv written: {len(screenings_df)} rows")
print(f"✓ transactions_clean.csv written: {len(transactions_df)} rows")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED ✓")
print("=" * 60)
print("\nNotebook 01_data_validation.ipynb complete.")
print("Ready for 02_kyc_checks.ipynb")

✓ customers_clean.csv written: 5120 rows
✓ id_verifications_clean.csv written: 3420 rows
✓ documents_clean.csv written: 7906 rows
✓ screenings_clean.csv written: 5037 rows
✓ transactions_clean.csv written: 5114 rows

ALL CHECKS PASSED ✓

Notebook 01_data_validation.ipynb complete.
Ready for 02_kyc_checks.ipynb
